# 📱 HyperNova 7/24 Mobil Veri Toplayıcı — Süper Hızlı APK Derleyici (Google Colab)

Bu Google Colab defteri, bilgisayarınıza **Android Studio kurmadan**, doğrudan Google bulut sunucularında **HyperNova Standalone APK** dosyasını 1 dakikada derler ve otomatik olarak bilgisayarınıza/telefonunuza indirir.

---
### 🚀 Nasıl Çalıştırılır?
1. Üst menüden **Çalışma Zamanı (Runtime) ➔ Tümünü Çalıştır (Run All)** veya `Ctrl + F9` tuşuna basın.
2. Yaklaşık 1 dakika sonra derlenen **`HyperNova-v1.0.0.apk`** dosyası tarayıcınız tarafından **otomatik olarak indirilecektir**.

## 🛠️ Adım 1: Android SDK ve Gradle Derleme Motorunu Kur (~30 saniye)

In [ ]:
import os
import shutil

print('⏳ [1/4] Android SDK 34 ve Java 17 ortamı hazırlanıyor...')

# 1. Gerekli Paketleri Kur
!sudo apt-get update -qq
!sudo apt-get install -y -qq openjdk-17-jdk wget unzip git

# 2. Android SDK Command-Line Tools Kurulumu
!mkdir -p /root/android-sdk/cmdline-tools
if not os.path.exists('/root/android-sdk/cmdline-tools/latest'):
    !wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/cmdline-tools.zip
    !unzip -q /tmp/cmdline-tools.zip -d /root/android-sdk/cmdline-tools
    !mv /root/android-sdk/cmdline-tools/cmdline-tools /root/android-sdk/cmdline-tools/latest
    !rm -f /tmp/cmdline-tools.zip
    # Lisansları onayla ve SDK 34'ü indir
    !yes | /root/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null 2>&1
    !/root/android-sdk/cmdline-tools/latest/bin/sdkmanager "platforms;android-34" "build-tools;34.0.0" > /dev/null 2>&1

# 3. Gradle 8.2 Kurulumu
if not os.path.exists('/opt/gradle-8.2.1'):
    !wget -q https://services.gradle.org/distributions/gradle-8.2.1-bin.zip -O /tmp/gradle.zip
    !unzip -q /tmp/gradle.zip -d /opt
    !rm -f /tmp/gradle.zip

print('✅ Android derleme motoru hazır!')

## 📦 Adım 2: Proje Kaynak Kodunu Çek ve Hazırla

In [ ]:
%cd /content
!rm -rf /content/HyperNova_Source
print('⏳ [2/4] HyperNova kaynak kodları alınıyor...')
!git clone https://github.com/emirgocuk/HyperNova.git /content/HyperNova_Source

# local.properties dosyasını oluştur (SDK yolunu göster)
with open('/content/HyperNova_Source/android/local.properties', 'w') as f:
    f.write('sdk.dir=/root/android-sdk\n')

print('✅ Kaynak kodlar ve Android modülü hazırlandı.')

## 🔨 Adım 3: APK'yı Derle (Gradle Native Build)

In [ ]:
import os
print('🚀 [3/4] APK Derleme Başlatılıyor...')
%cd /content/HyperNova_Source/android

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['ANDROID_HOME'] = '/root/android-sdk'

!/opt/gradle-8.2.1/bin/gradle assembleDebug --no-daemon
print('✅ Derleme başarıyla tamamlandı!')

## ⬇️ Adım 4: Derlenen APK'yı İndir

In [ ]:
import glob
import shutil
from google.colab import files

print('📦 [4/4] APK dosyası hazırlanıyor...')
apk_paths = glob.glob('/content/HyperNova_Source/android/app/build/outputs/apk/debug/*.apk')

if apk_paths:
    output_apk = '/content/HyperNova-v1.0.0.apk'
    shutil.copyfile(apk_paths[0], output_apk)
    print(f'🎉 TEBRİKLER! APK BAŞARIYLA ÜRETİLDİ:\n📁 {output_apk}\n')
    print('⬇️ APK dosyanız şimdi bilgisayarınıza/telefonunuza indiriliyor...')
    files.download(output_apk)
else:
    print('❌ APK bulunamadı. Lütfen derleme loglarını kontrol edin.')